# Day 3 · Write the test set before the system

**First-principles question:** How will you know the system works, before you build it?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import json
from collections import Counter
import matplotlib.pyplot as plt

INTENTS = ["billing", "outage", "speed", "cancel", "upgrade", "relocation", "other"]
CASES = ROOT / "data" / "acme" / "cases.jsonl"
customers = {c["id"]: c for c in json.loads((ROOT / "data" / "acme" / "customers.json").read_text())}
print(len(customers), "customers;", CASES)

## 1 · Load the cases
`cases.jsonl` has one JSON object per line ("JSON Lines").

In [ ]:
# TODO: read CASES line by line (CASES.read_text(encoding="utf-8").splitlines()),
#       skip empty lines, and turn each line into a dict with json.loads
cases = []
print(len(cases), "cases")
cases[0] if cases else None

## 2 · Validate every case
A test set with mistakes in it is worse than no test set.

In [ ]:
REQUIRED = {"id", "customer_id", "message", "expected_intent", "must_include", "must_not", "needs_human"}

def problems(case):
    """A list of problems with one case. An empty list means the case is valid."""
    found = []
    missing = REQUIRED - set(case)
    if missing:
        found.append(f"missing keys: {sorted(missing)}")
    # TODO: add a problem when...
    #   - expected_intent is not one of INTENTS
    #   - customer_id is not a key of `customers`
    #   - must_include or must_not is not a list of strings
    #   - needs_human is not True or False (use isinstance(x, bool))
    return found

bad = {c.get("id", "?"): problems(c) for c in cases if problems(c)}
dupes = [i for i, n in Counter(c.get("id") for c in cases).items() if n > 1]
print(bad or "Every case is valid ✅", "| duplicate ids:", dupes or "none")

## 3 · What are you testing?

In [ ]:
counts = Counter(c.get("expected_intent") for c in cases)
plt.figure(figsize=(7, 3))
plt.bar(INTENTS, [counts[i] for i in INTENTS], color="#2153c4")
plt.axhline(3, color="#c4362c", linestyle="--", label="at least 3 each")
plt.title(f"{len(cases)} test cases per intent"); plt.legend(); plt.show()
print("tags:", Counter(tag for c in cases for tag in c.get("tags", [])))

## 4 · Add your own cases
Open `data/acme/cases.jsonl` in VS Code and add one line per case (copy an existing line, then change it).
Ideas: an angry customer · Hinglish (`"tags": ["hinglish"]`) · two problems in one email · no customer details ·
a prompt-injection attempt (`"tags": ["injection"]`) · a legal threat (needs a human) · a polite thank-you (`other`).
Then re-run this notebook from the top.

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert problems({"id": "x"}), "problems() should report missing keys"
ok = {"id": "t", "customer_id": "AC1001", "message": "hi", "expected_intent": "billing",
      "must_include": ["Priya"], "must_not": [], "needs_human": False}
assert problems(ok) == [], f"A valid case was rejected: {problems(ok)}"
assert problems(dict(ok, expected_intent="weather")), "Unknown intents must be reported"
assert problems(dict(ok, customer_id="AC9999")), "Unknown customers must be reported"
assert problems(dict(ok, must_include="Priya")), "must_include must be a list"
assert problems(dict(ok, needs_human="no")), "needs_human must be True or False"
assert not bad and not dupes, "Fix the invalid or duplicate cases first"
assert len(cases) >= 30, f"Add more cases: {len(cases)} of 30"
low = [i for i in INTENTS if counts[i] < 3]
assert not low, f"These intents need at least 3 cases: {low}"
assert sum("hinglish" in c.get("tags", []) for c in cases) >= 3, "Tag at least 3 cases 'hinglish'"
assert any("injection" in c.get("tags", []) for c in cases), "Tag at least 1 case 'injection'"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")